# MedicalNLP-RAG — Phase 4C v2
## Evidence Grounding / Faithfulness

Phase 4B froze the final evidence strategy as:

`BM25 + MedCPT → RRF → MedCPT Cross-Encoder → Top-5 PubMed documents`

This notebook evaluates whether generated biomedical claims are actually supported by the evidence supplied to the answer generator. **No new Qwen answers are generated.**

### v2 design
The original prototype selected the maximum NLI entailment over all available evidence windows. Because different evidence conditions contain different numbers of candidate windows, this can inflate support scores. v2 separates evidence selection from faithfulness assessment:

1. split generated `ideal_answer` into deterministic claims;
2. construct evidence units available to each answer;
3. rank claim–evidence relevance with `ncbi/MedCPT-Cross-Encoder`;
4. keep a fixed **Top-3** evidence units per claim;
5. apply `pritamdeka/PubMedBERT-MNLI-MedNLI` only to those Top-3 units;
6. use **MedCPT rank-1 evidence** for the primary grounding metric;
7. use Top-3 support only as a sensitivity analysis.

### Conditions
- `gold_evidence`: Phase 4A answer + expert BioASQ snippets.
- `retrieved_rag`: Phase 4A answer + frozen reranked Top-10 documents.
- `document_top5`: Phase 4B answer + **frozen final Top-5 evidence strategy**.

### Primary metric
`top1_supported`: the MedCPT rank-1 evidence unit has NLI argmax label `entailment`.

No post-hoc entailment threshold is selected. MedCPT and NLI scores are automatic model-based signals, **not calibrated clinical probabilities**.

The 860-question internal test remains locked.

## 0. Configuration

In [ ]:
PROJECT_NAME = "MedicalNLP_RAG"
SEED = 42

FROZEN_EVIDENCE_STRATEGY = "document_top5"
ANALYSIS_CONDITIONS = ["gold_evidence", "retrieved_rag", "document_top5"]

MEDCPT_MODEL = "ncbi/MedCPT-Cross-Encoder"
MEDCPT_MAX_LENGTH = 512
MEDCPT_BATCH_SIZE = 48
GROUNDING_TOP_M = 3

NLI_MODEL = "pritamdeka/PubMedBERT-MNLI-MedNLI"
NLI_MAX_LENGTH = 512
NLI_BATCH_SIZE = 48

EVIDENCE_WINDOW_SENTENCES = 2
EVIDENCE_WINDOW_STRIDE = 1
MAX_WINDOWS_PER_DOCUMENT = 12
MIN_SENTENCE_CHARS = 20

RUN_SMOKE_TEST = True
RUN_FULL_SCORING = False
SMOKE_PER_TYPE = 3

CREATE_HUMAN_AUDIT_SAMPLE = True
HUMAN_AUDIT_PER_TYPE_PER_BAND = 10

ALLOW_INTERNAL_TEST_EVALUATION = False

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Dependency check

In [ ]:
import sys, subprocess, importlib, importlib.util
from packaging.version import Version

def get_version(name):
    try:
        return getattr(importlib.import_module(name), "__version__", "0")
    except Exception:
        return None

transformers_version = get_version("transformers")
need_install = (
    transformers_version is None
    or Version(transformers_version) < Version("4.45.0")
    or importlib.util.find_spec("scipy") is None
)

if need_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.45.0", "accelerate>=1.0.0", "scipy>=1.10.0"
    ])

print("Dependency check complete.")

## 3. Imports and paths

In [ ]:
from pathlib import Path
import re, gc, json, random, hashlib, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.stats import spearmanr
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path(f"/content/drive/MyDrive/{PROJECT_NAME}")
SPLIT_DIR = ROOT / "data" / "bioasq14_splits"
PUBMED_CACHE_DIR = ROOT / "data" / "pubmed_cache"
PHASE3C_DIR = ROOT / "results" / "phase3C_reranking"
PHASE4A_DIR = ROOT / "results" / "phase4A_answer_generation"
PHASE4B_DIR = ROOT / "results" / "phase4B_evidence_filtering"

PHASE4C_DIR = ROOT / "results" / "phase4C_grounding_v2"
TABLE_DIR = PHASE4C_DIR / "tables"
FIG_DIR = PHASE4C_DIR / "figures"
SCORE_CACHE_DIR = PHASE4C_DIR / "score_cache"
AUDIT_DIR = PHASE4C_DIR / "human_audit"
LOG_DIR = PHASE4C_DIR / "logs"
for p in [PHASE4C_DIR, TABLE_DIR, FIG_DIR, SCORE_CACHE_DIR, AUDIT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

VALIDATION_JSON = SPLIT_DIR / "validation.json"
CORPUS_CSV = PUBMED_CACHE_DIR / "pubmed_closed_corpus.csv"
FROZEN_RUN = PHASE3C_DIR / "tables" / "RRF_MedCPT_CrossEncoder_top100.jsonl"
FROZEN_EVIDENCE_JSON = PHASE4B_DIR / "frozen_evidence_strategy.json"
PHASE4A_GENERATIONS = PHASE4A_DIR / "generations" / "validation_862q_3conditions.jsonl"
PHASE4B_GENERATIONS = PHASE4B_DIR / "generations" / "validation_862q_2filtered_conditions.jsonl"
PHASE4B_PER_QUESTION_SCORES = PHASE4B_DIR / "tables" / "phase4B_answer_scores_per_question.csv"

for required in [VALIDATION_JSON, CORPUS_CSV, FROZEN_RUN, FROZEN_EVIDENCE_JSON,
                 PHASE4A_GENERATIONS, PHASE4B_GENERATIONS]:
    assert required.exists(), required

print("Phase 4C v2 output:", PHASE4C_DIR)

## 4. Locked-test safety

In [ ]:
if ALLOW_INTERNAL_TEST_EVALUATION:
    raise RuntimeError("Internal test must remain locked during Phase 4C v2.")
print("INTERNAL TEST REMAINS LOCKED.")

## 5. Verify frozen evidence strategy

In [ ]:
with open(FROZEN_EVIDENCE_JSON, "r", encoding="utf-8") as f:
    frozen_evidence = json.load(f)

display(pd.DataFrame([frozen_evidence]))
assert frozen_evidence["frozen_evidence_strategy"] == FROZEN_EVIDENCE_STRATEGY
assert frozen_evidence["internal_test_evaluated"] is False
print("Frozen evidence strategy verified:", FROZEN_EVIDENCE_STRATEGY)

## 6. Load validation questions

In [ ]:
with open(VALIDATION_JSON, "r", encoding="utf-8") as f:
    validation_questions = json.load(f)["questions"]
assert len(validation_questions) == 862

rows = []
for q in validation_questions:
    rows.append({
        "id": str(q["id"]),
        "type": str(q["type"]).strip().lower(),
        "body": str(q["body"]).strip(),
        "exact_answer_gold": q.get("exact_answer"),
        "ideal_answer_gold": q.get("ideal_answer"),
        "snippets_gold": q.get("snippets", []),
        "documents_gold": q.get("documents", []),
    })
val_df = pd.DataFrame(rows)
assert val_df["id"].nunique() == 862
assert set(val_df["type"]) == {"yesno", "factoid", "list", "summary"}
question_lookup = {str(q["id"]): q for q in validation_questions}
display(val_df["type"].value_counts())

## 7. Load PubMed corpus and frozen rankings

In [ ]:
corpus_df = pd.read_csv(CORPUS_CSV, dtype={"pmid": str})
corpus_df["pmid"] = corpus_df["pmid"].astype(str).str.replace(r"\.0$", "", regex=True)
corpus_df["title"] = corpus_df["title"].fillna("").astype(str).str.strip()
corpus_df["abstract"] = corpus_df["abstract"].fillna("").astype(str).str.strip()
corpus_lookup = (
    corpus_df.drop_duplicates("pmid").set_index("pmid")[["title", "abstract"]].to_dict(orient="index")
)

frozen_rankings = {}
with open(FROZEN_RUN, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        frozen_rankings[str(rec["id"])] = [str(x) for x in rec["ranked_pmids"]]
assert len(frozen_rankings) == 862
assert all(len(v) >= 10 for v in frozen_rankings.values())
print("Corpus records:", len(corpus_lookup))
print("Frozen rankings:", len(frozen_rankings))

## 8. Load existing Phase 4A and 4B answers

In [ ]:
def read_jsonl(path):
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                out.append(json.loads(line))
    return pd.DataFrame(out)

phase4a_df = read_jsonl(PHASE4A_GENERATIONS)
phase4b_df = read_jsonl(PHASE4B_GENERATIONS)
answers_df = pd.concat([
    phase4a_df[phase4a_df["condition"].isin(["gold_evidence", "retrieved_rag"])].copy(),
    phase4b_df[phase4b_df["condition"] == "document_top5"].copy(),
], ignore_index=True)
answers_df["id"] = answers_df["id"].astype(str)
assert len(answers_df) == 862 * 3

display(answers_df.groupby(["condition", "type"]).size().unstack(fill_value=0))
display(answers_df.groupby("condition")["parse_valid"].mean().rename("json_valid_rate"))

## 9. Save experiment environment

In [ ]:
import transformers

environment = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "BioASQ14b",
    "validation_questions": 862,
    "internal_test_evaluated": False,
    "frozen_evidence_strategy": FROZEN_EVIDENCE_STRATEGY,
    "analysis_conditions": ANALYSIS_CONDITIONS,
    "medcpt_model": MEDCPT_MODEL,
    "grounding_top_m": GROUNDING_TOP_M,
    "nli_model": NLI_MODEL,
    "medcpt_max_length": MEDCPT_MAX_LENGTH,
    "nli_max_length": NLI_MAX_LENGTH,
    "evidence_window_sentences": EVIDENCE_WINDOW_SENTENCES,
    "evidence_window_stride": EVIDENCE_WINDOW_STRIDE,
    "max_windows_per_document": MAX_WINDOWS_PER_DOCUMENT,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
}
with open(LOG_DIR / "environment.json", "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2, ensure_ascii=False)
print(json.dumps(environment, indent=2, ensure_ascii=False))

# Part A — Evidence units

Gold evidence uses BioASQ expert snippets. Retrieved conditions use title + overlapping two-sentence abstract windows. Evidence construction is deterministic and independent of Phase 4C v1.

## 10. Sentence splitting and PMID extraction

In [ ]:
def split_sentences(text):
    text = " ".join(str(text).split()).strip()
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
    return [p.strip() for p in parts if len(p.strip()) >= MIN_SENTENCE_CHARS]

def extract_pmid(ref):
    s = str(ref).strip().rstrip("/")
    m = re.search(r"(\d+)(?:\D*)$", s)
    return m.group(1) if m else None

## 11. Evidence builders

In [ ]:
def make_windows(sentences, window_size=EVIDENCE_WINDOW_SENTENCES, stride=EVIDENCE_WINDOW_STRIDE):
    if not sentences:
        return []
    if len(sentences) <= window_size:
        return [" ".join(sentences)]
    return [
        " ".join(sentences[start:start + window_size])
        for start in range(0, len(sentences) - window_size + 1, stride)
    ]

def build_retrieved_units(qid, condition):
    top_k = 10 if condition == "retrieved_rag" else 5 if condition == "document_top5" else None
    if top_k is None:
        raise ValueError(condition)
    units = []
    for document_rank, pmid in enumerate(frozen_rankings[str(qid)][:top_k], start=1):
        rec = corpus_lookup.get(str(pmid))
        if rec is None:
            continue
        title, abstract = rec["title"], rec["abstract"]
        sentences = split_sentences(abstract)
        if not sentences:
            sentences = [abstract] if abstract else [title] if title else []
        for window_index, window in enumerate(make_windows(sentences)[:MAX_WINDOWS_PER_DOCUMENT], start=1):
            text = f"Title: {title}\nEvidence: {window}" if title else f"Evidence: {window}"
            units.append({
                "id": str(qid), "condition": condition,
                "evidence_id": f"{condition}|{qid}|{pmid}|{document_rank}|{window_index}",
                "pmid": str(pmid), "document_rank": int(document_rank),
                "window_index": int(window_index), "evidence_text": text,
            })
    return units

def build_gold_units(qid):
    q = question_lookup[str(qid)]
    units = []
    for idx, snippet in enumerate(q.get("snippets", []), start=1):
        if isinstance(snippet, dict):
            text = str(snippet.get("text", "")).strip()
            document = str(snippet.get("document", "")).strip()
        else:
            text, document = str(snippet).strip(), ""
        if not text:
            continue
        pmid = extract_pmid(document)
        units.append({
            "id": str(qid), "condition": "gold_evidence",
            "evidence_id": f"gold_evidence|{qid}|{idx}",
            "pmid": str(pmid) if pmid else "", "document_rank": np.nan,
            "window_index": int(idx), "evidence_text": f"Evidence: {text}",
        })
    return units

## 12. Build all evidence units

In [ ]:
evidence_rows = []
for qid in tqdm(val_df["id"], desc="Building evidence units"):
    evidence_rows.extend(build_gold_units(qid))
    evidence_rows.extend(build_retrieved_units(qid, "retrieved_rag"))
    evidence_rows.extend(build_retrieved_units(qid, "document_top5"))

evidence_units_df = pd.DataFrame(evidence_rows)
assert evidence_units_df["evidence_id"].is_unique
evidence_units_df.to_csv(TABLE_DIR / "grounding_evidence_units.csv", index=False)

unit_summary = evidence_units_df.groupby("condition").agg(
    n_units=("evidence_id", "size"), n_questions=("id", "nunique")
).reset_index()
unit_summary["mean_units_per_question"] = unit_summary["n_units"] / unit_summary["n_questions"]
display(unit_summary)

# Part B — Deterministic claim extraction

## 13. Claim splitter

In [ ]:
def split_answer_claims(text):
    text = " ".join(str(text).split()).strip()
    if not text:
        return []
    claims = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
    clean = [c.strip() for c in claims if len(c.strip()) >= 8]
    return clean if clean else [text]

## 14. Build generated-claim table

In [ ]:
claim_rows = []
for row in answers_df.itertuples(index=False):
    valid = bool(row.parse_valid)
    ideal_answer = str(row.ideal_answer_pred or "").strip() if valid else ""
    for claim_index, claim in enumerate(split_answer_claims(ideal_answer), start=1):
        claim_rows.append({
            "id": str(row.id), "type": row.type, "condition": row.condition,
            "parse_valid": valid, "claim_index": int(claim_index),
            "claim_id": f"{row.condition}|{row.id}|{claim_index}", "claim_text": claim,
        })
claims_df = pd.DataFrame(claim_rows)
claims_df.to_csv(TABLE_DIR / "generated_answer_claims.csv", index=False)
print("Total claims:", len(claims_df))
display(claims_df.groupby(["condition", "type"]).agg(n_claims=("claim_id", "size"), n_questions=("id", "nunique")))

## 15. Claim-count audit

In [ ]:
claim_count_summary = (
    claims_df.groupby(["condition", "type", "id"]).size().rename("n_claims").reset_index()
    .groupby(["condition", "type"])["n_claims"].describe().round(2)
)
claim_count_summary.to_csv(TABLE_DIR / "claim_count_summary.csv")
display(claim_count_summary)

# Part C — Claim × evidence relevance pairs

## 16. Pair builder

In [ ]:
evidence_group_lookup = {
    key: group.copy() for key, group in evidence_units_df.groupby(["id", "condition"], sort=False)
}

def build_relevance_pairs(selected_ids=None):
    selected_claims = claims_df if selected_ids is None else claims_df[claims_df["id"].isin({str(x) for x in selected_ids})]
    pair_rows = []
    for claim in tqdm(selected_claims.itertuples(index=False), total=len(selected_claims), desc="Building MedCPT pairs"):
        group = evidence_group_lookup.get((str(claim.id), str(claim.condition)))
        if group is None or len(group) == 0:
            continue
        for ev in group.itertuples(index=False):
            raw = "\t".join([str(claim.claim_id), str(claim.claim_text), str(ev.evidence_id), str(ev.evidence_text)])
            pair_rows.append({
                "pair_key": hashlib.sha256(raw.encode("utf-8")).hexdigest(),
                "id": str(claim.id), "type": claim.type, "condition": claim.condition,
                "claim_id": claim.claim_id, "claim_index": int(claim.claim_index), "claim_text": claim.claim_text,
                "evidence_id": ev.evidence_id, "pmid": str(ev.pmid), "document_rank": ev.document_rank,
                "window_index": ev.window_index, "evidence_text": ev.evidence_text,
            })
    return pd.DataFrame(pair_rows)

# Part D — MedCPT relevance ranking

## 17. GPU check

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU before running Phase 4C v2.")
DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

## 18. Model load/unload helpers

In [ ]:
def load_medcpt_model():
    global medcpt_tokenizer, medcpt_model
    medcpt_tokenizer = AutoTokenizer.from_pretrained(MEDCPT_MODEL)
    medcpt_model = AutoModelForSequenceClassification.from_pretrained(MEDCPT_MODEL).to(DEVICE)
    medcpt_model.eval()
    print("Loaded:", MEDCPT_MODEL, "num_labels=", medcpt_model.config.num_labels)

def unload_medcpt():
    global medcpt_tokenizer, medcpt_model
    if "medcpt_model" in globals(): del medcpt_model
    if "medcpt_tokenizer" in globals(): del medcpt_tokenizer
    gc.collect(); torch.cuda.empty_cache()
    print("MedCPT unloaded.")

def unload_nli():
    global nli_tokenizer, nli_model
    if "nli_model" in globals(): del nli_model
    if "nli_tokenizer" in globals(): del nli_tokenizer
    gc.collect(); torch.cuda.empty_cache()
    print("NLI unloaded.")

load_medcpt_model()

## 19. Resumable MedCPT scorer

In [ ]:
MEDCPT_SCORE_CACHE = SCORE_CACHE_DIR / "claim_evidence_medcpt_scores.csv"

def load_medcpt_cache():
    if not MEDCPT_SCORE_CACHE.exists():
        return pd.DataFrame(columns=["pair_key", "medcpt_score"])
    return pd.read_csv(MEDCPT_SCORE_CACHE, dtype={"pair_key": str}).drop_duplicates("pair_key", keep="last").reset_index(drop=True)

def append_medcpt_scores(records):
    if not records: return
    chunk = pd.DataFrame(records)
    if MEDCPT_SCORE_CACHE.exists():
        chunk.to_csv(MEDCPT_SCORE_CACHE, mode="a", header=False, index=False)
    else:
        chunk.to_csv(MEDCPT_SCORE_CACHE, index=False)

def score_medcpt_pairs(pair_df):
    cache = load_medcpt_cache(); completed = set(cache["pair_key"].astype(str))
    todo = pair_df[~pair_df["pair_key"].isin(completed)].reset_index(drop=True)
    print("Pairs requested:", len(pair_df), "| cached:", len(pair_df)-len(todo), "| remaining:", len(todo))
    buffer = []
    for start in tqdm(range(0, len(todo), MEDCPT_BATCH_SIZE), desc="MedCPT claim-evidence relevance"):
        batch = todo.iloc[start:start+MEDCPT_BATCH_SIZE]
        enc = medcpt_tokenizer(
            text=batch["claim_text"].tolist(), text_pair=batch["evidence_text"].tolist(),
            padding=True, truncation=True, max_length=MEDCPT_MAX_LENGTH, return_tensors="pt"
        )
        enc = {k:v.to(DEVICE) for k,v in enc.items()}
        with torch.no_grad(): logits = medcpt_model(**enc).logits.float()
        if logits.ndim == 1: scores = logits
        elif logits.ndim == 2 and logits.shape[1] == 1: scores = logits[:,0]
        elif logits.ndim == 2 and logits.shape[1] == 2: scores = logits[:,1]
        else: raise RuntimeError(f"Unexpected MedCPT logits shape: {tuple(logits.shape)}")
        for key, score in zip(batch["pair_key"], scores.detach().cpu().numpy()):
            buffer.append({"pair_key": str(key), "medcpt_score": float(score)})
        if len(buffer) >= 3000:
            append_medcpt_scores(buffer); buffer=[]
    append_medcpt_scores(buffer)
    result = pair_df.merge(load_medcpt_cache(), on="pair_key", how="left", validate="one_to_one")
    assert result["medcpt_score"].notna().all()
    return result

## 20. Select fixed MedCPT Top-3 per claim

In [ ]:
def select_medcpt_top_m(scored_pairs):
    parts=[]
    for claim_id, group in scored_pairs.groupby("claim_id", sort=False):
        g=group.copy(); g["document_rank_sort"]=g["document_rank"].fillna(10000)
        g=(g.sort_values(["medcpt_score","document_rank_sort","window_index","evidence_id"],
                         ascending=[False,True,True,True]).head(GROUNDING_TOP_M).reset_index(drop=True))
        g["medcpt_rank"]=np.arange(len(g))+1
        parts.append(g.drop(columns=["document_rank_sort"]))
    return pd.concat(parts, ignore_index=True)

# Part E — Biomedical NLI on fixed Top-3 only

## 21. Load NLI model and verify label mapping

In [ ]:
def load_nli_model():
    global nli_tokenizer, nli_model, raw_id2label, label2id
    global CONTRADICTION_IDX, ENTAILMENT_IDX, NEUTRAL_IDX
    nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
    nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE)
    nli_model.eval()
    cfg = {int(k):str(v).lower() for k,v in nli_model.config.id2label.items()}
    expected={"contradiction","entailment","neutral"}
    if set(cfg.values()) == expected:
        raw_id2label=cfg
    else:
        if nli_model.config.num_labels != 3: raise RuntimeError("Unexpected NLI label configuration.")
        raw_id2label={0:"contradiction",1:"entailment",2:"neutral"}
    label2id={v:k for k,v in raw_id2label.items()}
    CONTRADICTION_IDX=label2id["contradiction"]; ENTAILMENT_IDX=label2id["entailment"]; NEUTRAL_IDX=label2id["neutral"]
    print("Loaded:", NLI_MODEL); print("NLI labels:", raw_id2label)

def run_nli_sanity_check():
    premises=["Aspirin inhibits platelet aggregation and is used as an antiplatelet drug."]*2
    hypotheses=["Aspirin has antiplatelet effects.", "Aspirin has no antiplatelet effects."]
    enc=nli_tokenizer(text=premises,text_pair=hypotheses,padding=True,truncation=True,max_length=NLI_MAX_LENGTH,return_tensors="pt")
    enc={k:v.to(DEVICE) for k,v in enc.items()}
    with torch.no_grad(): logits=nli_model(**enc).logits.float()
    probs=torch.softmax(logits,dim=-1).cpu().numpy()
    df=pd.DataFrame({
        "hypothesis":hypotheses,
        "p_contradiction":probs[:,CONTRADICTION_IDX],
        "p_entailment":probs[:,ENTAILMENT_IDX],
        "p_neutral":probs[:,NEUTRAL_IDX],
        "predicted_label":[raw_id2label[int(i)] for i in probs.argmax(axis=1)]
    })
    display(df)
    assert df.loc[0,"p_entailment"] > df.loc[0,"p_contradiction"]
    assert df.loc[1,"p_contradiction"] > df.loc[1,"p_entailment"]
    print("NLI sanity check passed.")

## 22. Resumable NLI scorer

In [ ]:
NLI_SCORE_CACHE = SCORE_CACHE_DIR / "top3_claim_evidence_nli_scores.csv"

def load_nli_cache():
    if not NLI_SCORE_CACHE.exists():
        return pd.DataFrame(columns=["pair_key","p_contradiction","p_entailment","p_neutral","nli_label"])
    return pd.read_csv(NLI_SCORE_CACHE,dtype={"pair_key":str}).drop_duplicates("pair_key",keep="last").reset_index(drop=True)

def append_nli_scores(records):
    if not records: return
    chunk=pd.DataFrame(records)
    if NLI_SCORE_CACHE.exists(): chunk.to_csv(NLI_SCORE_CACHE,mode="a",header=False,index=False)
    else: chunk.to_csv(NLI_SCORE_CACHE,index=False)

def score_nli_selected(selected_df):
    cache=load_nli_cache(); completed=set(cache["pair_key"].astype(str))
    todo=selected_df[~selected_df["pair_key"].isin(completed)].reset_index(drop=True)
    print("Selected pairs:",len(selected_df),"| cached:",len(selected_df)-len(todo),"| remaining:",len(todo))
    buffer=[]
    for start in tqdm(range(0,len(todo),NLI_BATCH_SIZE),desc="Biomedical NLI on MedCPT Top-3"):
        batch=todo.iloc[start:start+NLI_BATCH_SIZE]
        enc=nli_tokenizer(text=batch["evidence_text"].tolist(),text_pair=batch["claim_text"].tolist(),
                          padding=True,truncation=True,max_length=NLI_MAX_LENGTH,return_tensors="pt")
        enc={k:v.to(DEVICE) for k,v in enc.items()}
        with torch.no_grad(): logits=nli_model(**enc).logits.float()
        probs=torch.softmax(logits,dim=-1).cpu().numpy(); pred=probs.argmax(axis=1)
        for i,key in enumerate(batch["pair_key"]):
            buffer.append({
                "pair_key":str(key), "p_contradiction":float(probs[i,CONTRADICTION_IDX]),
                "p_entailment":float(probs[i,ENTAILMENT_IDX]), "p_neutral":float(probs[i,NEUTRAL_IDX]),
                "nli_label":raw_id2label[int(pred[i])]
            })
        if len(buffer)>=3000: append_nli_scores(buffer); buffer=[]
    append_nli_scores(buffer)
    result=selected_df.merge(load_nli_cache(),on="pair_key",how="left",validate="one_to_one")
    assert result["p_entailment"].notna().all()
    return result

# Part F — 12-question smoke test

## 23. Build deterministic smoke sample

In [ ]:
smoke_parts=[]
for qtype in sorted(val_df["type"].unique()):
    smoke_parts.append(val_df[val_df["type"]==qtype].sample(n=SMOKE_PER_TYPE,random_state=SEED))
smoke_df=(pd.concat(smoke_parts,ignore_index=True).sort_values(["type","id"]).reset_index(drop=True))
smoke_ids=smoke_df["id"].astype(str).tolist()
display(smoke_df[["id","type","body"]])

## 24. Smoke: MedCPT relevance → fixed Top-3

In [ ]:
smoke_top3=None
if RUN_SMOKE_TEST:
    smoke_pairs=build_relevance_pairs(smoke_ids)
    print("Smoke relevance pairs:",len(smoke_pairs))
    smoke_medcpt=score_medcpt_pairs(smoke_pairs)
    smoke_top3=select_medcpt_top_m(smoke_medcpt)
    smoke_top3.to_csv(TABLE_DIR/"smoke_medcpt_top3_evidence.csv",index=False)
    display(smoke_top3[["id","type","condition","claim_index","claim_text","medcpt_rank","medcpt_score","pmid","document_rank","evidence_text"]].head(40))

## 25. Smoke: switch to NLI and run sanity check

In [ ]:
if RUN_SMOKE_TEST:
    unload_medcpt()
    load_nli_model()
    run_nli_sanity_check()

## 26. Smoke: NLI on fixed Top-3

In [ ]:
smoke_nli_scores=None
if RUN_SMOKE_TEST:
    smoke_nli_scores=score_nli_selected(smoke_top3)
    smoke_nli_scores.to_csv(TABLE_DIR/"smoke_top3_nli_scores.csv",index=False)
    display(smoke_nli_scores[[
        "id","type","condition","claim_index","claim_text","medcpt_rank","medcpt_score","pmid","document_rank",
        "p_entailment","p_neutral","p_contradiction","nli_label","evidence_text"
    ]].sort_values(["type","id","condition","claim_index","medcpt_rank"]))

## 27. Derive v2 claim-level metrics

In [ ]:
def derive_claim_grounding_v2(selected_nli_df):
    rows=[]
    for claim_id, group in selected_nli_df.groupby("claim_id",sort=False):
        g=group.sort_values("medcpt_rank").copy(); top1=g.iloc[0]
        best_ent=g.loc[g["p_entailment"].idxmax()]
        rows.append({
            "id":str(top1["id"]),"type":top1["type"],"condition":top1["condition"],"claim_id":claim_id,
            "claim_index":int(top1["claim_index"]),"claim_text":top1["claim_text"],"n_selected_evidence":int(len(g)),
            "top1_medcpt_score":float(top1["medcpt_score"]),"top1_pmid":str(top1["pmid"]),
            "top1_document_rank":top1["document_rank"],"top1_evidence_id":top1["evidence_id"],"top1_evidence_text":top1["evidence_text"],
            "top1_entailment":float(top1["p_entailment"]),"top1_neutral":float(top1["p_neutral"]),
            "top1_contradiction":float(top1["p_contradiction"]),"top1_nli_label":top1["nli_label"],
            "top1_supported":bool(top1["nli_label"]=="entailment"),"top1_contradicted":bool(top1["nli_label"]=="contradiction"),
            "top3_supported":bool((g["nli_label"]=="entailment").any()),
            "top3_contradicted_any":bool((g["nli_label"]=="contradiction").any()),
            "top3_entailment_count":int((g["nli_label"]=="entailment").sum()),
            "top3_max_entailment":float(g["p_entailment"].max()),"top3_max_contradiction":float(g["p_contradiction"].max()),
            "top3_best_entailment_rank":int(best_ent["medcpt_rank"]),"top3_best_entailment_pmid":str(best_ent["pmid"]),
            "top3_best_entailment_text":best_ent["evidence_text"],
        })
    return pd.DataFrame(rows)

## 28. Inspect smoke grounding

In [ ]:
smoke_claim_grounding=None
if smoke_nli_scores is not None:
    smoke_claim_grounding=derive_claim_grounding_v2(smoke_nli_scores)
    smoke_claim_grounding.to_csv(TABLE_DIR/"smoke_claim_grounding_v2.csv",index=False)
    display(smoke_claim_grounding.sort_values(["type","id","condition","claim_index"])[[
        "id","type","condition","claim_index","claim_text","top1_medcpt_score","top1_pmid","top1_document_rank",
        "top1_entailment","top1_neutral","top1_contradiction","top1_nli_label","top1_supported","top3_supported",
        "top3_max_entailment","top1_evidence_text"
    ]])

# STOP POINT 4C-v2-SMOKE

Stop here before full scoring. Check that:
- MedCPT rank-1 evidence is semantically related to the claim;
- the NLI sanity test passed;
- Top-1 labels are plausible in representative cases;
- NLI softmax scores are finite and sum approximately to 1;
- `top3_supported` is treated only as sensitivity analysis.

Do **not** change `GROUNDING_TOP_M=3` or add an entailment threshold after inspecting smoke output.

Keep `RUN_FULL_SCORING=False` until review.

# Part G — Full validation scoring

## 29. Full MedCPT relevance ranking

In [ ]:
def unload_nli():
    global nli_model
    global nli_tokenizer

    if "nli_model" in globals():
        del nli_model

    if "nli_tokenizer" in globals():
        del nli_tokenizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("NLI model unloaded from GPU.")

In [ ]:
RUN_FULL_SCORING = True
print("RUN_FULL_SCORING:", RUN_FULL_SCORING)

In [ ]:
full_top3=None
if RUN_FULL_SCORING:
    if "nli_model" in globals(): unload_nli()
    if "medcpt_model" not in globals(): load_medcpt_model()
    full_pairs=build_relevance_pairs(None)
    print("Full relevance pairs:",len(full_pairs))
    full_medcpt=score_medcpt_pairs(full_pairs)
    full_top3=select_medcpt_top_m(full_medcpt)
    full_top3.to_csv(TABLE_DIR/"medcpt_top3_evidence_per_claim.csv",index=False)
    print("Selected Top-3 rows:",len(full_top3))
else:
    print("RUN_FULL_SCORING=False — full MedCPT ranking not started.")

## 30. Full NLI scoring

In [ ]:
full_nli_scores=None
if RUN_FULL_SCORING:
    unload_medcpt()
    load_nli_model()
    run_nli_sanity_check()
    full_nli_scores=score_nli_selected(full_top3)
    full_nli_scores.to_csv(TABLE_DIR/"top3_claim_evidence_nli_scores.csv",index=False)
    print("Full Top-3 NLI scoring complete:",len(full_nli_scores))

## 31. Full claim-level grounding

In [ ]:
claim_grounding_df=None
if full_nli_scores is not None:
    claim_grounding_df=derive_claim_grounding_v2(full_nli_scores)
    claim_grounding_df.to_csv(TABLE_DIR/"claim_grounding_v2.csv",index=False)
    print("Grounded claims:",len(claim_grounding_df))
    display(claim_grounding_df.head())

# Part H — Question-level grounding

## 32. Build per-question grounding metrics

In [ ]:
question_grounding_df=None
if claim_grounding_df is not None:
    base=answers_df[["id","type","condition","parse_valid"]].copy()
    agg=(claim_grounding_df.groupby(["id","type","condition"]).agg(
        n_claims=("claim_id","size"),
        mean_top1_entailment=("top1_entailment","mean"),
        mean_top1_contradiction=("top1_contradiction","mean"),
        top1_supported_claim_rate=("top1_supported","mean"),
        top1_contradiction_rate=("top1_contradicted","mean"),
        mean_top3_max_entailment=("top3_max_entailment","mean"),
        top3_supported_claim_rate=("top3_supported","mean"),
        top3_contradiction_any_rate=("top3_contradicted_any","mean"),
    ).reset_index())
    agg["fully_top1_supported"]=(agg["top1_supported_claim_rate"]==1.0).astype(float)
    agg["fully_top3_supported"]=(agg["top3_supported_claim_rate"]==1.0).astype(float)
    question_grounding_df=base.merge(agg,on=["id","type","condition"],how="left",validate="one_to_one")
    zero_cols=["n_claims","mean_top1_entailment","mean_top1_contradiction","top1_supported_claim_rate",
               "top1_contradiction_rate","mean_top3_max_entailment","top3_supported_claim_rate",
               "top3_contradiction_any_rate","fully_top1_supported","fully_top3_supported"]
    question_grounding_df[zero_cols]=question_grounding_df[zero_cols].fillna(0)
    question_grounding_df["n_claims"]=question_grounding_df["n_claims"].astype(int)
    question_grounding_df.to_csv(TABLE_DIR/"question_grounding_v2.csv",index=False)
    display(question_grounding_df.head())

## 33. Overall grounding summary

In [ ]:
grounding_summary=None
if question_grounding_df is not None:
    pooled=claim_grounding_df.groupby("condition").agg(
        pooled_top1_supported_claim_rate=("top1_supported","mean"),
        pooled_top3_supported_claim_rate=("top3_supported","mean")
    )
    grounding_summary=(question_grounding_df.groupby("condition").agg(
        n_questions=("id","size"),json_valid_rate=("parse_valid","mean"),mean_claims_per_question=("n_claims","mean"),
        mean_top1_entailment=("mean_top1_entailment","mean"),mean_top1_contradiction=("mean_top1_contradiction","mean"),
        question_mean_top1_supported_claim_rate=("top1_supported_claim_rate","mean"),
        fully_top1_supported_answer_rate=("fully_top1_supported","mean"),
        question_mean_top1_contradiction_rate=("top1_contradiction_rate","mean"),
        mean_top3_max_entailment=("mean_top3_max_entailment","mean"),
        question_mean_top3_supported_claim_rate=("top3_supported_claim_rate","mean"),
        fully_top3_supported_answer_rate=("fully_top3_supported","mean")
    ).join(pooled).reset_index())
    grounding_summary.to_csv(TABLE_DIR/"grounding_summary_v2.csv",index=False)
    display(grounding_summary.sort_values("question_mean_top1_supported_claim_rate",ascending=False))

## 34. Grounding by question type

In [ ]:
grounding_by_type=None
if question_grounding_df is not None:
    grounding_by_type=(question_grounding_df.groupby(["condition","type"]).agg(
        n=("id","size"),mean_top1_entailment=("mean_top1_entailment","mean"),
        top1_supported_claim_rate=("top1_supported_claim_rate","mean"),
        fully_top1_supported_answer_rate=("fully_top1_supported","mean"),
        top1_contradiction_rate=("top1_contradiction_rate","mean"),
        top3_supported_claim_rate=("top3_supported_claim_rate","mean")
    ).reset_index())
    grounding_by_type.to_csv(TABLE_DIR/"grounding_by_question_type_v2.csv",index=False)
    display(grounding_by_type)

# Part I — Paired bootstrap

## 35. Bootstrap helper and comparisons

In [ ]:
def paired_bootstrap(df, condition_a, condition_b, metric, qtype=None, n_boot=5000, seed=SEED):
    work=df if qtype is None else df[df["type"]==qtype]
    a=work[work["condition"]==condition_a][["id",metric]].rename(columns={metric:"a"})
    b=work[work["condition"]==condition_b][["id",metric]].rename(columns={metric:"b"})
    merged=a.merge(b,on="id",validate="one_to_one"); va=merged["a"].to_numpy(float); vb=merged["b"].to_numpy(float)
    rng=np.random.default_rng(seed); n=len(merged); deltas=[]
    for _ in range(n_boot):
        idx=rng.integers(0,n,size=n); deltas.append(np.mean(va[idx])-np.mean(vb[idx]))
    deltas=np.asarray(deltas)
    return {"condition_a":condition_a,"condition_b":condition_b,"metric":metric,"qtype":qtype or "all","n":int(n),
            "mean_delta":float(deltas.mean()),"ci95_low":float(np.quantile(deltas,.025)),"ci95_high":float(np.quantile(deltas,.975)),
            "bootstrap_probability_delta_gt_0":float(np.mean(deltas>0))}

grounding_bootstrap_df=None
if question_grounding_df is not None:
    comparisons=[("document_top5","retrieved_rag"),("gold_evidence","document_top5")]
    metrics=["mean_top1_entailment","top1_supported_claim_rate","fully_top1_supported","top1_contradiction_rate",
             "mean_top3_max_entailment","top3_supported_claim_rate","fully_top3_supported"]
    rows=[]
    for a,b in comparisons:
        for metric in metrics: rows.append(paired_bootstrap(question_grounding_df,a,b,metric))
    grounding_bootstrap_df=pd.DataFrame(rows)
    grounding_bootstrap_df.to_csv(TABLE_DIR/"paired_bootstrap_grounding_v2.csv",index=False)
    display(grounding_bootstrap_df)

# Part J — Attribution diagnostics

## 36. Rank distribution and lowest-grounded claims

In [ ]:
if claim_grounding_df is not None:
    top5=claim_grounding_df[claim_grounding_df["condition"]=="document_top5"].copy()
    rank_dist=top5["top1_document_rank"].value_counts(dropna=False).sort_index().rename("n_claims").to_frame()
    rank_dist["proportion"]=rank_dist["n_claims"]/rank_dist["n_claims"].sum()
    rank_dist.to_csv(TABLE_DIR/"top1_medcpt_document_rank_distribution.csv")
    display(rank_dist)

    lowest=(top5.sort_values(["top1_supported","top1_entailment","top1_contradiction"],ascending=[True,True,False]).head(30))
    lowest.to_csv(TABLE_DIR/"lowest_grounded_claims_top5_v2.csv",index=False)
    display(lowest[["id","type","claim_text","top1_medcpt_score","top1_pmid","top1_document_rank","top1_entailment",
                    "top1_neutral","top1_contradiction","top1_nli_label","top1_supported","top3_supported","top1_evidence_text"]])

## 37. Highest-grounded claims

In [ ]:
if claim_grounding_df is not None:
    highest=(claim_grounding_df[claim_grounding_df["condition"]=="document_top5"]
             .sort_values("top1_entailment",ascending=False).head(30))
    highest.to_csv(TABLE_DIR/"highest_grounded_claims_top5_v2.csv",index=False)
    display(highest[["id","type","claim_text","top1_pmid","top1_document_rank","top1_entailment","top1_nli_label",
                     "top3_supported","top1_evidence_text"]])

# Part K — Figures

## 38. Overall Top-1 / Top-3 grounding figure

In [ ]:
if grounding_summary is not None:
    plot_df=grounding_summary[["condition","question_mean_top1_supported_claim_rate","fully_top1_supported_answer_rate",
                               "question_mean_top3_supported_claim_rate"]].set_index("condition")
    fig,ax=plt.subplots(figsize=(9,5.5)); plot_df.plot(kind="bar",ax=ax)
    ax.set_xlabel("Evidence condition"); ax.set_ylabel("Automatic grounding rate")
    ax.set_title("Evidence grounding: Top-1 primary and Top-3 sensitivity"); ax.set_ylim(0,1); ax.tick_params(axis="x",rotation=0)
    fig.tight_layout(); fig.savefig(FIG_DIR/"fig01_grounding_top1_top3.png",dpi=300,bbox_inches="tight"); plt.show()

## 39. Top-1 supported-claim rate by question type

In [ ]:
if grounding_by_type is not None:
    pivot=grounding_by_type.pivot(index="type",columns="condition",values="top1_supported_claim_rate")
    fig,ax=plt.subplots(figsize=(9,5.5)); pivot.plot(kind="bar",ax=ax)
    ax.set_xlabel("Question type"); ax.set_ylabel("Top-1 supported-claim rate")
    ax.set_title("Primary evidence grounding by BioASQ question type"); ax.set_ylim(0,1); ax.tick_params(axis="x",rotation=0)
    fig.tight_layout(); fig.savefig(FIG_DIR/"fig02_top1_supported_claim_rate_by_type.png",dpi=300,bbox_inches="tight"); plt.show()

# Part L — Grounding versus answer quality

## 40. Merge with Phase 4B per-question quality and compute Spearman associations

In [ ]:
grounding_quality_df=None
grounding_quality_spearman=None
if question_grounding_df is not None and PHASE4B_PER_QUESTION_SCORES.exists():
    answer_scores=pd.read_csv(PHASE4B_PER_QUESTION_SCORES,dtype={"id":str})
    answer_scores=answer_scores[answer_scores["condition"]=="document_top5"].copy()
    def task_score(row):
        if row["type"]=="yesno": return row["yesno_correct"]
        if row["type"]=="factoid": return row["factoid_mrr"]
        if row["type"]=="list": return row["list_f1"]
        if row["type"]=="summary": return row["rougeL_f"]
        return np.nan
    answer_scores["task_score"]=answer_scores.apply(task_score,axis=1)
    frozen=question_grounding_df[question_grounding_df["condition"]=="document_top5"].copy()
    grounding_quality_df=frozen.merge(answer_scores[["id","type","task_score"]],on=["id","type"],validate="one_to_one")
    grounding_quality_df.to_csv(TABLE_DIR/"grounding_vs_answer_quality_v2.csv",index=False)

    rows=[]; signals=["mean_top1_entailment","top1_supported_claim_rate","mean_top1_contradiction","top3_supported_claim_rate"]
    for qtype in ["all","yesno","factoid","list","summary"]:
        subset=grounding_quality_df if qtype=="all" else grounding_quality_df[grounding_quality_df["type"]==qtype]
        for signal in signals:
            rho,p=spearmanr(subset[signal],subset["task_score"],nan_policy="omit")
            rows.append({"qtype":qtype,"grounding_signal":signal,"n":int(len(subset)),"spearman_rho":float(rho),"p_value":float(p)})
    grounding_quality_spearman=pd.DataFrame(rows)
    grounding_quality_spearman.to_csv(TABLE_DIR/"grounding_quality_spearman_v2.csv",index=False)
    display(grounding_quality_spearman)

# Part M — Human grounding audit

## 41. Export stratified manual-audit sample

In [ ]:
human_audit_df=None
if CREATE_HUMAN_AUDIT_SAMPLE and claim_grounding_df is not None:
    pool=claim_grounding_df[claim_grounding_df["condition"]=="document_top5"].copy()
    pool["score_band"]=pd.cut(pool["top1_entailment"],bins=[-0.001,0.33,0.67,1.001],labels=["low","medium","high"],include_lowest=True)
    sampled=[]
    for (qtype,band),group in pool.groupby(["type","score_band"],observed=True):
        n=min(HUMAN_AUDIT_PER_TYPE_PER_BAND,len(group))
        if n: sampled.append(group.sample(n=n,random_state=SEED))
    if sampled:
        human_audit_df=pd.concat(sampled,ignore_index=True).sort_values(["type","score_band","id"]).reset_index(drop=True)
        qmap=val_df.set_index("id")["body"].to_dict(); human_audit_df["question"]=human_audit_df["id"].map(qmap)
        human_audit_df["human_support_label"]=""; human_audit_df["human_confidence"]=""; human_audit_df["human_notes"]=""
        cols=["id","type","score_band","question","claim_text","top1_medcpt_score","top1_pmid","top1_document_rank",
              "top1_evidence_text","top1_entailment","top1_neutral","top1_contradiction","top1_nli_label","top1_supported",
              "top3_supported","human_support_label","human_confidence","human_notes"]
        human_audit_df=human_audit_df[cols]
        human_audit_df.to_csv(AUDIT_DIR/"human_grounding_audit_v2.csv",index=False)
        print("Human audit rows:",len(human_audit_df)); display(human_audit_df.head(20))

# Part N — Freeze Phase 4C v2 method

## 42. Freeze grounding method

In [ ]:
grounding_method={
    "dataset":"BioASQ14b","validation_questions":862,"internal_test_evaluated":False,
    "frozen_evidence_strategy":FROZEN_EVIDENCE_STRATEGY,
    "claim_extraction":"deterministic sentence segmentation of generated ideal_answer",
    "evidence_selection":{
        "model":MEDCPT_MODEL,"score_use":"ranking only; no threshold","fixed_top_m":GROUNDING_TOP_M,
        "retrieved_evidence_window_sentences":EVIDENCE_WINDOW_SENTENCES,"retrieved_evidence_window_stride":EVIDENCE_WINDOW_STRIDE,
        "max_windows_per_document":MAX_WINDOWS_PER_DOCUMENT,"gold_evidence_unit":"BioASQ expert snippet"
    },
    "nli":{
        "model":NLI_MODEL,"premise":"selected evidence unit","hypothesis":"generated claim",
        "primary_support_rule":"MedCPT rank-1 evidence has NLI argmax label entailment",
        "sensitivity_support_rule":"at least one fixed MedCPT Top-3 evidence unit has NLI argmax label entailment",
        "entailment_threshold_selected":False,
        "score_interpretation":"automatic uncalibrated NLI signal; not clinical probability"
    },
    "primary_claim_metric":"top1_supported","primary_question_metric":"top1_supported_claim_rate",
    "sensitivity_metric":"top3_supported_claim_rate",
    "candidate_confidence_signals_for_phase4D":[
        "mean_top1_entailment","top1_supported_claim_rate","mean_top1_contradiction","top1_contradiction_rate","top3_supported_claim_rate"
    ]
}
with open(PHASE4C_DIR/"frozen_grounding_method_v2.json","w",encoding="utf-8") as f:
    json.dump(grounding_method,f,indent=2,ensure_ascii=False)
print(json.dumps(grounding_method,indent=2,ensure_ascii=False))

## 43. Phase 4C v2 report

In [ ]:
if grounding_summary is not None:
    phase4c_report={
        "dataset":"BioASQ14b","validation_questions":862,"internal_test_evaluated":False,
        "frozen_evidence_strategy":FROZEN_EVIDENCE_STRATEGY,"grounding_method":grounding_method,
        "automatic_grounding_summary":grounding_summary.to_dict(orient="records"),
        "grounding_by_question_type":grounding_by_type.to_dict(orient="records") if grounding_by_type is not None else None,
        "paired_bootstrap":grounding_bootstrap_df.to_dict(orient="records") if grounding_bootstrap_df is not None else None,
        "human_audit_created":bool(human_audit_df is not None),
        "human_audit_rows":int(len(human_audit_df)) if human_audit_df is not None else 0,
        "note":"MedCPT and NLI scores are automatic model-based grounding signals and are not calibrated clinical probabilities."
    }
    with open(PHASE4C_DIR/"phase4C_v2_report.json","w",encoding="utf-8") as f:
        json.dump(phase4c_report,f,indent=2,ensure_ascii=False)
    print(json.dumps(phase4c_report,indent=2,ensure_ascii=False))

# STOP POINT 4C-v2-FULL

After the full run, send:
1. `grounding_summary_v2`
2. `grounding_by_question_type_v2`
3. `paired_bootstrap_grounding_v2`
4. Top-1 MedCPT document-rank distribution
5. lowest-grounded frozen Top-5 claims
6. both grounding figures
7. `grounding_quality_spearman_v2`
8. `frozen_grounding_method_v2.json`
9. `phase4C_v2_report.json`

Keep `human_grounding_audit_v2.csv` for optional expert validation.

Next: **Phase 4D — Confidence Estimation / Calibration**.

## 44. Final safety check

In [ ]:
if not ALLOW_INTERNAL_TEST_EVALUATION:
    print("INTERNAL TEST REMAINS LOCKED.")
else:
    raise RuntimeError("Do not unlock the internal test during Phase 4C v2.")
print("Phase 4C v2 output:",PHASE4C_DIR)